# wf_v4.1 Nov 16 run (diary Nov 7, task 7)
Code: GitHub `TriDang2025/attribute-extraction`, branch `wf-v4.1`, folder `step1a/`. Runner `scripts/nov16_v41.py`, helpers `s1a/nov16_v41.py`.

Seven stages, in this order: **Step 1b → loop 2 probe → Step 3 naming of the loop-2 clusters → duplicate check → Step 4 v4.1 (18 chains) → contamination test → task 13 (placement test)**. Data: dev50 and dev50b only.

- After every stage the results are committed and pushed (`results/wf_v41/nov16/<tag>/`), and `state.json` records the stage as done.
- **After a crash or a disconnect:** run the cells again from the top (skip cell 2 if vLLM is still installed). Finished stages are skipped, and finished calls inside a stage are read from the cache.
- Model: chosen in cell 3 from `config/wf_v41_models.json` (`qwen32b` = Qwen2.5-32B-Instruct-AWQ at the v4 revision; `llama8b` = Llama-3.1-8B-Instruct, bf16). Same prompts and stages for both; temperature 0, guided JSON from each job's schema. Runtime: **A100 (40 GB or more)**.
- Each setting writes its own run folder (`results/wf_v41/nov16/<setting>/`). With `llama8b`, task 13 runs the base chain with Llama too (4 more calls) instead of copying the v4 Qwen answers.
- Llama is gated: the HF account of `HF_TOKEN` must have accepted the Llama 3.1 license.
- Colab Secrets `HF_TOKEN` and `GH_TOKEN`, notebook access on. Never put a token in a cell.

In [ ]:
#@title 1. Tokens, clone/pull wf-v4.1, git identity
GIT_NAME = "Your Name"  #@param {type:"string"}
GIT_EMAIL = "you@users.noreply.github.com"  #@param {type:"string"}
BRANCH = "wf-v4.1"  #@param {type:"string"}
import os, subprocess
from google.colab import userdata
from huggingface_hub import login

hf_token = (userdata.get("HF_TOKEN") or "").strip()
if hf_token:
    login(token=hf_token)
del hf_token
os.environ["GH_TOKEN"] = (userdata.get("GH_TOKEN") or "").strip()
def run(cmd, **kw): subprocess.run(cmd, check=True, **kw)
run(["git", "config", "--global", "user.name", GIT_NAME])
run(["git", "config", "--global", "user.email", GIT_EMAIL])
run(["git", "config", "--global", "credential.helper",
     '!f() { echo username=token; echo "password=$GH_TOKEN"; }; f'])
REPO = "/content/attribute-extraction"
if not os.path.exists(REPO):
    run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/TriDang2025/attribute-extraction.git", REPO])
else:
    run(["git", "-C", REPO, "checkout", "-q", BRANCH])
    run(["git", "-C", REPO, "pull", "-q", "--rebase", "--autostash"])
%cd /content/attribute-extraction/step1a
!git log -1 --format="code at %h (%an, %ar): %s"

**Cell 2 note:** the install prints dependency-resolver warnings; ignore them. Then **Runtime > Restart session**, rerun cell 1 and skip cell 2.
gliner and sentence-transformers are installed with `--no-deps` so they do not replace the torch / transformers that vLLM needs. Cell 4 checks that they load.

In [ ]:
#@title 2. Install (pinned; a few minutes; then restart the session and skip this cell)
!pip install -q vllm==0.33.0
!pip install -q --no-deps gliner==0.2.29 sentence-transformers==5.6.0
!pip install -q sentencepiece

In [ ]:
#@title 3. Model setting (config/wf_v41_models.json) and GPU check
SETTING = "qwen32b"  #@param ["qwen32b", "llama8b"]
import sys
sys.path.insert(0, "/content/attribute-extraction/step1a")
from s1a import nov16_v41 as NV
SET = NV.model_setting(SETTING)
print(SETTING, SET)
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), "no GPU: change the runtime type to A100"
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU memory {gb:.1f} GiB")
assert gb >= SET["min_gpu_gb"], f"only {gb:.1f} GiB: {SETTING} needs {SET['min_gpu_gb']} GiB (an A100)"

In [ ]:
#@title 4. Data and the loop-2 models (MultiWOZ zip + ids from HF, GLiNER, deberta-v3-base, bge-m3; pinned revisions)
import json, os
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download
MODELS = Path("/content/models")
os.environ["CHK_MODELS"] = str(MODELS)          # read by chunking/chk/config.py and s1a/spans.py at import
DATA = Path("/content/attribute-extraction/chunking/data")
for f in ("MULTIWOZ2.4.zip", "conversation_ids.txt"):
    if not (DATA / f).exists():
        hf_hub_download("database-lab/attribute-extraction-data", f"chunking/{f}", repo_type="dataset", local_dir=DATA.parent)
snapshot_download("urchade/gliner_medium-v2.1", revision="40ec419335d09393f298636f471328b722c6da9e", local_dir=MODELS / "gliner_medium-v2.1")
snapshot_download("microsoft/deberta-v3-base", revision="8ccc9b6f36199bec6961081d44eb72fb3f7353f3", local_dir=MODELS / "deberta-v3-base",
                  allow_patterns=["config.json", "spm.model", "tokenizer_config.json"])
snapshot_download("BAAI/bge-m3", revision="5617a9f61b028005a4858fdac845db406aefb181", local_dir=MODELS / "bge-m3")
cfg_path = MODELS / "gliner_medium-v2.1" / "gliner_config.json"   # point GLiNER at the local backbone, as on the laptop
cfg = json.loads(cfg_path.read_text())
cfg["model_name"] = str(MODELS / "deberta-v3-base")
cfg_path.write_text(json.dumps(cfg, indent=2))
import sys
sys.path.insert(0, "/content/attribute-extraction/step1a"); sys.path.insert(0, "/content/attribute-extraction/step1a/scripts")
from s1a import spans as SP
g = SP.load()
print("GLiNER ok:", g.inference(["I need a hotel with free wifi."], ["internet access"], threshold=0.3))
del g

In [ ]:
#@title 5. Load the model (gpu_memory_utilization leaves room for GLiNER and bge-m3 of loop 2)
MODEL, REVISION = SET["model"], SET["revision"]   # from the setting of cell 3
GPU_UTIL = 0.80  #@param {type:"number"}
TAG = SETTING   # run folder results/wf_v41/nov16/<TAG>
MAX_LEN = 32768
import vllm
from vllm import LLM, SamplingParams
print("vllm", vllm.__version__, "| model", MODEL, "| revision", REVISION)
llm = LLM(model=MODEL, revision=REVISION, tokenizer_revision=REVISION, quantization=SET["quantization"],
          max_model_len=MAX_LEN, gpu_memory_utilization=GPU_UTIL)

In [ ]:
#@title 6. Backend: temperature 0, guided JSON per job schema, batches, cache per job (s1a/nov16_v41.VllmBackend)
BATCH = 64  #@param {type:"integer"}
def sampling(max_tokens, schema):
    kw = dict(temperature=0, max_tokens=max_tokens)
    if schema is not None:
        try:
            from vllm.sampling_params import StructuredOutputsParams
            kw["structured_outputs"] = StructuredOutputsParams(json=schema)
        except ImportError:
            from vllm.sampling_params import GuidedDecodingParams
            kw["guided_decoding"] = GuidedDecodingParams(json=schema)
    return SamplingParams(**kw)
backend = NV.VllmBackend(llm, MODEL, REVISION, sampling, max_len=MAX_LEN, batch=BATCH, vllm_version=vllm.__version__)
backend.model_id

In [ ]:
#@title 7. Run all stages (each stage is committed and pushed when it ends; a rerun skips the finished ones)
import subprocess
import nov16_v41 as P
RUN = Path("results/wf_v41/nov16") / TAG
def push(stage):
    r = subprocess.run(["bash", "colab/sync.sh", "push-results", f"nov16 {TAG} {stage}", f"wf_v41/nov16/{TAG}"],
                       capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr[-500:])
state = P.run_all(RUN, backend, on_saved=push, copy_v4_base=SET["copy_v4_base"])
print("done:", state["done"])

In [ ]:
#@title 8. Summary: calls and tokens per stage, the time of each stage
print((RUN / "calls.md").read_text())
for s, info in state["info"].items():
    print(f"{s:14} {info['seconds'] / 60:6.1f} min")

**Then:** tell Claude "run done". On the laptop: `git pull`, then score Step 4 (`scripts/step4_v41.py --report --dir <run>/step4`), task 13 (`scripts/assemble_v4.py --score`) and the main claim (`scripts/mainclaim_v41.py --loop2 <run>/loop2/<set>/clusters.json`).